Create a webpage - make it cool

Enter a new sentence - it should be classified as either coming from prepared remarks, or Q&A
Use only responses to questions in your Q&A model

Build the naive bayes model

Show the naive bayes scores, etc…

## Load the cleaned dataset

In [1]:
import json
from pathlib import Path
corpus_path = next((p for p in [Path('data/corpus.json'), Path('corpus.json')] if p.exists()), None)
if corpus_path is None:
    try:
        from google.colab import files
        print('Upload data/corpus.json from the assignment package.')
        uploaded = files.upload()
        corpus = json.loads(uploaded[next(name for name in uploaded if name.endswith('.json'))])
    except ImportError:
        raise FileNotFoundError('Open in RemarksLab, or place corpus.json next to the notebook.')
else:
    corpus = json.loads(corpus_path.read_text(encoding='utf-8'))
print('Retained turns:', len(corpus))
print('Calls:', len({d['call_id'] for d in corpus}))
for d in corpus[:3]:
    print(d['call_id'], d['turn_id'], d['company'], d['speaker'], d['label'], d['split'])

Retained turns: 3230
Calls: 185
1 2 Fastenal Dray Schreiber prepared train
1 3 Fastenal Jeffery Watts prepared train
1 4 Fastenal Max Tunnicliff prepared train


## Check the split
set seed = 42, train:test = 8:2. Same company or conference won't be randomly separated into both sides.

In [2]:
from collections import Counter
print(Counter((d['split'], d['label']) for d in corpus))
for split in ['train', 'test']:
    print(split, 'calls:', len({d['call_id'] for d in corpus if d['split'] == split}))
assert not ({d['call_id'] for d in corpus if d['split']=='train'} & {d['call_id'] for d in corpus if d['split']=='test'})
assert not ({d['company'].lower() for d in corpus if d['split']=='train'} & {d['company'].lower() for d in corpus if d['split']=='test'})

Counter({('train', 'qa'): 2051, ('train', 'prepared'): 543, ('test', 'qa'): 506, ('test', 'prepared'): 130})
train calls: 148
test calls: 37


## Build multinomial Naive Bayes
`P(w|c)=(count(w,c)+1)/(N_c+|V|)`

`score(c)=ln(0.5)+Σ n(w)ln P(w|c)`

In [3]:
"""Transparent multinomial Naive Bayes; Python standard library only."""
from collections import Counter
import math, re, json
from pathlib import Path
LABELS=('prepared','qa')

def tokenize(text):
    return re.findall(r"[a-z]+(?:'[a-z]+)?",text.lower().replace('’',"'").replace('‘',"'"))

def train(corpus, alpha=1.0):
    if alpha<=0: raise ValueError('alpha must be positive')
    counts={c:Counter() for c in LABELS}
    for doc in corpus:
        if doc['split']=='train': counts[doc['label']].update(tokenize(doc['text']))
    vocabulary=set(counts['prepared'])|set(counts['qa'])
    return dict(alpha=alpha,counts={c:dict(counts[c]) for c in LABELS},
        totals={c:sum(counts[c].values()) for c in LABELS},vocabulary=sorted(vocabulary),
        priors={c:0.5 for c in LABELS})

def classify(text,model):
    tokens=tokenize(text);vocab=model['vocabulary'];known=lambda w:w in model['counts']['prepared'] or w in model['counts']['qa'];freq=Counter(t for t in tokens if known(t))
    scores={c:math.log(model['priors'][c]) for c in LABELS};rows=[]
    for word,n in freq.items():
        row=dict(word=word,n=n)
        for c in LABELS:
            prob=(model['counts'][c].get(word,0)+model['alpha'])/(model['totals'][c]+model['alpha']*len(vocab))
            row[c]=n*math.log(prob);scores[c]+=row[c]
        row['delta']=row['prepared']-row['qa'];rows.append(row)
    largest=max(scores.values());weights={c:math.exp(scores[c]-largest) for c in LABELS}
    probs={c:weights[c]/sum(weights.values()) for c in LABELS}
    tie=abs(scores['prepared']-scores['qa'])<1e-12
    return dict(label=None if not freq or tie else max(scores,key=scores.get),scores=scores,probabilities=probs,
        tokens=tokens,known=sum(freq.values()),unknown=[t for t in tokens if not known(t)],rows=rows)

def sentences(text):
    # Lightweight, deterministic segmentation. Protect common abbreviations.
    text=re.sub(r'\b(?:U\.S\.|U\.N\.|Mr\.|Mrs\.|Dr\.)',lambda m:m[0].replace('.','∯'),text)
    parts=re.split(r'(?<=[.!?])[”\"\']*\s+|\n\s*\n',text)
    return [p.replace('∯','.').strip() for p in parts if len(tokenize(p))>=4]

def evaluate(corpus,model):
    matrix={a:{b:0 for b in (*LABELS,'unclassified')} for a in LABELS};examples=[]
    train_sentences={tuple(tokenize(s)) for d in corpus if d['split']=='train' for s in sentences(d['text'])}
    removed=0
    for doc in corpus:
        if doc['split']!='test': continue
        for sentence in sentences(doc['text']):
            if tuple(tokenize(sentence)) in train_sentences: removed+=1;continue
            result=classify(sentence,model);pred=result['label'] or 'unclassified'
            matrix[doc['label']][pred]+=1
            examples.append(dict(text=sentence,actual=doc['label'],predicted=pred,source=doc['id'],call_id=doc.get('call_id'),speaker=doc.get('speaker')))
    n=sum(sum(row.values()) for row in matrix.values());correct=sum(matrix[c][c] for c in LABELS)
    recalls={c:matrix[c][c]/sum(matrix[c].values()) for c in LABELS}
    return dict(matrix=matrix,n=n,accuracy=correct/n,balanced_accuracy=sum(recalls.values())/2,
        recalls=recalls,majority_baseline=max(sum(row.values()) for row in matrix.values())/n,
        removed_duplicates=removed,examples=examples)



In [4]:
model = train(corpus)
print('Training tokens:', model['totals'])
print('Vocabulary:', len(model['vocabulary']))
print('Priors:', model['priors'])
for c in LABELS:
    print(c, Counter(model['counts'][c]).most_common(10))

Training tokens: {'prepared': 432382, 'qa': 394091}
Vocabulary: 15815
Priors: {'prepared': 0.5, 'qa': 0.5}
prepared [('the', 19723), ('and', 16172), ('to', 13961), ('of', 11092), ('our', 10943), ('in', 9493), ('we', 9167), ('a', 6707), ('year', 4702), ('for', 4569)]
qa [('the', 19169), ('and', 12777), ('we', 12377), ('to', 11464), ('that', 11335), ('of', 10265), ('in', 7943), ('a', 7723), ('is', 7072), ('i', 5393)]


## Enter a new sentence

In [5]:
new_sentence = 'Yeah, thanks for the question. I think what you are seeing is a bit of a timing issue, and let me explain that.'
r = classify(new_sentence, model)
print('Prediction:', r['label'])
print('Scores:', r['scores'])
print('Normalized model probabilities:', r['probabilities'])
print('Recognized:', r['known'], '/', len(r['tokens']))
print('Ignored words:', r['unknown'])
print('WORD             N     PREPARED LOG        QA LOG      DIFFERENCE')
for row in sorted(r['rows'], key=lambda row: abs(row['delta']), reverse=True):
    print(f"{row['word']:15} {row['n']:3} {row['prepared']:15.6f} {row['qa']:13.6f} {row['delta']:13.6f}")

Prediction: qa
Scores: {'prepared': -155.42552453838175, 'qa': -133.21336330776492}
Normalized model probabilities: {'prepared': 2.256217410957995e-10, 'qa': 0.9999999997743783}
Recognized: 23 / 23
Ignored words: []
WORD             N     PREPARED LOG        QA LOG      DIFFERENCE
yeah              1      -11.914376     -6.895405     -5.018971
think             1       -8.397868     -4.964407     -3.433460
bit               1       -9.579001     -6.578047     -3.000954
question          1       -9.100965     -6.455984     -2.644981
i                 1       -5.844408     -4.330641     -1.513768
what              1       -6.989541     -5.481190     -1.508350
you               1       -6.036640     -4.538423     -1.498217
seeing            1       -7.781880     -6.459095     -1.322785
explain           1      -10.240399    -11.537389      1.296989
that              1       -4.875300     -3.587944     -1.287356
issue             1      -10.240399     -8.991858     -1.248542
is            

## Evaluate on held-out calls

In [6]:
evaluation = evaluate(corpus, model)
print('Confusion matrix: actual rows / predicted columns')
print(json.dumps(evaluation['matrix'], indent=2))
for key in ['n','accuracy','balanced_accuracy','majority_baseline','recalls','removed_duplicates']:
    print(key, evaluation[key])
print('\nExamples of errors:')
for example in [e for e in evaluation['examples'] if e['actual'] != e['predicted']][:4]:
    print(example)

Confusion matrix: actual rows / predicted columns
{
  "prepared": {
    "prepared": 4701,
    "qa": 809,
    "unclassified": 0
  },
  "qa": {
    "prepared": 477,
    "qa": 4890,
    "unclassified": 0
  }
}
n 10877
accuracy 0.8817688700928565
balanced_accuracy 0.8821497881285005
majority_baseline 0.5065735037234531
recalls {'prepared': 0.8531760435571688, 'qa': 0.9111235326998323}
removed_duplicates 67

Examples of errors:
{'text': 'As we move through October, the 2 known unknowns that we wrote about in our March shareholder letter.', 'actual': 'prepared', 'predicted': 'qa', 'source': 'turn-101', 'call_id': 3, 'speaker': 'Ted Pick'}
{'text': 'And the salutary effects of common sense regulation are kicking in.', 'actual': 'prepared', 'predicted': 'qa', 'source': 'turn-101', 'call_id': 3, 'speaker': 'Ted Pick'}
{'text': 'With that, I will turn it over to Sharon to take you through the quarter in greater detail.', 'actual': 'prepared', 'predicted': 'qa', 'source': 'turn-101', 'call_id': 3

## Verify the formula and edge cases

In [7]:
import math
for c in LABELS:
    total = sum((model['counts'][c].get(w,0)+1)/(model['totals'][c]+len(model['vocabulary'])) for w in model['vocabulary'])
    assert abs(total-1) < 1e-10
for text in ['', '!!!', '你好', 'zxqv blorf']:
    assert classify(text, model)['label'] is None
print('Normalization and empty/unknown-input checks passed.')

Normalization and empty/unknown-input checks passed.
